# Narkomfin Building Graph Analysis - PART 1

## 1. Import the needed libraries -

In [1]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## 2. Check the TopologicPy Version

In [2]:
print(Helper.Version())

The version that you are using (0.9.50) is EQUAL TO the latest version available on PyPI.


## 3. Set your renderer:

In [3]:
renderer = "vscode"

## 4. Utility functions to reset the face dictionaries and transfer dictionaries by key

In [4]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == "face_id":
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts[str(value)]
            f = Topology.SetDictionary(f, d)


## 5. Load Type K floor plan

In [11]:
from pathlib import Path

HERE = Path.cwd()  # VS Code sets CWD to the notebook's folder

BREP_TYPE_K = HERE.parent / '02_graph_analysis' / 'output' / 'L1_narkomfin_type_k_face.brep'

plan_k = Topology.ByBREPPath(str(BREP_TYPE_K))
print('Type K plan loaded.')


Type K plan loaded.


## 6. Show the geometry

In [12]:
Topology.Show(plan_k,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor='white',
              edgeWidth=3,
              showVertices=False,
              backgroundColor='black',
              width=800,
              height=500,
              renderer=renderer)


## 7. Create grid overlay for Type K

In [13]:
b_r_k   = Wire.BoundingRectangle(plan_k)
d_k     = Topology.Dictionary(b_r_k)
xmin_k  = Dictionary.ValueAtKey(d_k, 'xmin')
xmax_k  = Dictionary.ValueAtKey(d_k, 'xmax')
ymin_k  = Dictionary.ValueAtKey(d_k, 'ymin')
ymax_k  = Dictionary.ValueAtKey(d_k, 'ymax')
width_k = Dictionary.ValueAtKey(d_k, 'width')
length_k= Dictionary.ValueAtKey(d_k, 'length')
uRange_k = list(range(0, int(width_k)+2, 2))
vRange_k = list(range(0, int(length_k)+2, 2))
grid_k  = Grid.EdgesByDistances(plan_k, clip=True, uRange=uRange_k, vRange=vRange_k)


## 8. Show the geometry and the grid

In [14]:
Topology.Show(plan_k, grid_k,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor='grey',
              edgeWidth=3,
              showVertices=False,
              backgroundColor='black',
              width=800,
              height=500,
              renderer=renderer)


## 9. Slice the floor plan with its grid to create a topologic shell

In [15]:
shell_k = Topology.Slice(plan_k, grid_k)
faces_k = Topology.Faces(shell_k)
for i, f in enumerate(faces_k):
    d = Dictionary.ByKeyValue('face_id', 'tk_face_'+str(i+1))
    f = Topology.SetDictionary(f, d)

print(f'Type K shell: {len(faces_k)} faces')


Type K shell: 302 faces


## 10. Show the shell

In [16]:
Topology.Show(shell_k,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=0.9,
              edgeColor='black',
              edgeWidth=3,
              showVertices=False,
              backgroundColor='black',
              width=800,
              height=500,
              renderer=renderer)


## 11. Derive navigation and analysis graphs from the shell

In [ ]:
# Note: Graph nodes automatically inherit the dictionaries of the entities they come from
navigation_graph_k = Graph.ByTopology(shell_k, direct=False, viaSharedTopologies=True)
analysis_graph_k   = Graph.ByTopology(shell_k)


## 12. Derive and store the analysis graph vertices

In [ ]:
g_verts_k = Graph.Vertices(analysis_graph_k)


## 13. Show the analysis graph

In [ ]:
Topology.Show(analysis_graph_k,
              camera=[0,0,6],
              vertexSize=4,
              vertexColor='red',
              edgeColor='lightgrey',
              backgroundColor='black',
              width=800,
              height=500,
              renderer=renderer)


## 14. Spatial Intelligence through Graph Analysis

### b. Shortest Path (Use navigation graph)

In [ ]:
import time

start_k = Vertex.ByCoordinates(xmin_k+2, ymax_k-2, 0)
end_k   = Vertex.ByCoordinates(xmax_k-2, ymin_k+2, 0)
crg_k   = Graph.CompiledRoutingGraph(navigation_graph_k, precomputeTurns=False)
t0      = time.time()
shortest_path_k = Graph.ShortestPath(crg_k, vertexA=start_k, vertexB=end_k)
print('Type K — Shortest Path:', round(time.time()-t0, 2), 's')
straight_path_k = Wire.Straighten(shortest_path_k, host=plan_k)
print('  Original length:', round(Wire.Length(shortest_path_k), 2))
print('  Straightened length:', round(Wire.Length(straight_path_k), 2))
for edge in Topology.Edges(shortest_path_k):
    edge = Topology.SetDictionary(edge, Dictionary.ByKeysValues(['width','color'], [7,'red']))
for edge in Topology.Edges(straight_path_k):
    edge = Topology.SetDictionary(edge, Dictionary.ByKeysValues(['width','color'], [7,'blue']))


In [ ]:
Topology.Show(plan_k, shortest_path_k, straight_path_k,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColorKey='color',
              edgeWidthKey='width',
              backgroundColor='black',
              width=800,
              height=500,
              renderer=renderer)


### c. Closeness Centrality/Integration
* Closeness centrality is a graph metric that quantifies how close a node is to all other nodes by taking the reciprocal of the sum of its shortest path distances to every other node in the network.
* In space syntax, closeness centrality corresponds to global integration, measuring how spatially accessible or topologically shallow a space is within a configuration, thereby indicating its potential for movement flow and encounter density.

In [ ]:
centrality_list_k = Graph.ClosenessCentrality(analysis_graph_k, colorScale='thermal')


* Transfer the information from the graph back to the shell

In [ ]:
reset_dictionaries(shell_k)
faces_k = Topology.Faces(shell_k)
_ = transfer_dicts_by_key(faces_k, g_verts_k, 'face_id')


In [ ]:
Topology.Show(faces_k,
              faceColorKey='cc_color',
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor='black',
              width=800,
              height=500,
              renderer=renderer)


### d. Betweenness Centrality/Choice
* Betweenness centrality measures how often a node lies on the shortest paths between other nodes.

In [ ]:
centrality_list_k = Graph.BetweennessCentrality(analysis_graph_k, normalize=True, colorScale='thermal')


* Transfer the information from the graph back to the shell

In [ ]:
reset_dictionaries(shell_k)
faces_k = Topology.Faces(shell_k)
_ = transfer_dicts_by_key(faces_k, g_verts_k, 'face_id')


In [ ]:
Topology.Show(faces_k,
              faceColorKey='bc_color',
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor='black',
              width=800,
              height=500,
              renderer=renderer)
